# Train and upload your first Classic model

This example trains a model with the Numerai v6 data and saves a `.pkl` file for Model Uploads. The upload lets Numerai run your model on new live data. Predictions from the new data are not scored until the new submissions open.

In [ ]:
!python --version
!pip install -q numerapi 'pandas>=2.3,<3' pyarrow lightgbm cloudpickle==3.1.1

## 1. Train a starter model

The small feature set keeps this example approachable. `feature_country` is text, so this first model uses the numeric features only. You can build a stronger model with more features and your own research.

In [ ]:
from pathlib import Path
import json

import lightgbm as lgb
import pandas as pd
from numerapi import NumerAPI

DATA_VERSION = "v6.0"
napi = NumerAPI()
train_path = Path(DATA_VERSION) / "train.parquet"
features_path = Path(DATA_VERSION) / "features.json"

for path in (train_path, features_path):
    if not path.exists():
        napi.download_dataset(str(path))

with features_path.open() as file:
    feature_sets = json.load(file)["feature_sets"]
features = [name for name in feature_sets["small"] if name != "feature_country"]
assert features, "The small feature set has no numeric features"

train = pd.read_parquet(train_path, columns=features + ["target"])
train = train.dropna(subset=["target"])
assert not train.empty, "The training file has no mature target rows"
train = train.sample(n=min(200_000, len(train)), random_state=7)

model = lgb.LGBMRegressor(
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    colsample_bytree=0.1,
    verbosity=-1,
)
model.fit(train[features], train["target"])
print(f"Trained on {len(train):,} rows and {len(features)} features")

## 2. Save the model for upload

Model Uploads calls `predict(live_features, live_benchmark_models)` when new live data arrives. This starter model uses only the features; the benchmark predictions are available for more advanced models.

In [ ]:
import cloudpickle


def predict(live_features: pd.DataFrame, live_benchmark_models: pd.DataFrame) -> pd.DataFrame:
    predictions = model.predict(live_features[features])
    ranked = pd.Series(predictions, index=live_features.index).rank(pct=True)
    return ranked.to_frame("prediction")


model_filename = "tutorial_basics.pkl"
with open(model_filename, "wb") as file:
    cloudpickle.dump(predict, file)
print(f"Saved {model_filename}")

## 3. Check the pickle on live data

Load the saved file and call it with the current live features. Model Uploads will run the same `predict` function for future live data.

In [ ]:
live_path = Path(DATA_VERSION) / "live.parquet"
if not live_path.exists():
    napi.download_dataset(str(live_path))

live_features = pd.read_parquet(live_path, columns=features)
with open(model_filename, "rb") as file:
    saved_predict = cloudpickle.load(file)
example_predictions = saved_predict(live_features, pd.DataFrame(index=live_features.index))
assert example_predictions.index.equals(live_features.index)
assert example_predictions["prediction"].notna().all()
assert example_predictions["prediction"].between(0, 1).all()
example_predictions.head()

## 4. Upload your model

Download `tutorial_basics.pkl` and use **Upload Model** in the Tournament basics tutorial. Keep your Python environment compatible with the Model Uploads runtime. New-data predictions are not scored until the new submissions open.

In [ ]:
try:
    from google.colab import files
    files.download(model_filename)
except ImportError:
    print(f"Your model is saved locally as {model_filename}")